# Train color_strong seeds 10–11 (Kaggle, fresh)

Trains the last two **color_strong** encoders (Shapes3D) from scratch to the
50-epoch cut. Seeds 0–9 are already done. **No checkpoints needed** — fresh
training starts from each seed's random init, so there is nothing to upload and
no `Add Input` step.

## One-time setup
1. **Settings → Accelerator → `GPU T4 x2`** (or single T4 — both work; x2 runs
   both seeds at once).
2. **Settings → Internet → On** (clone + install + Shapes3D download).

## Run order
Sections 1–4 top to bottom (clone, install, data), then **section 5** (train).
Section 5 is idempotent + resumable: it skips a seed once its `backbone.pt`
exists and resumes a partial seed from its own `last_ckpt.pt`, so **re-run it
after a session ends** to continue. **Save Version** every few hours so
`/kaggle/working` persists (Kaggle sessions cap ~12 h).

Wall-clock: ~5 GPU-h per full Shapes3D encoder; two seeds ÷ 2 GPUs ≈ ~5 h — fits
one session. If a session ends mid-run, just re-run section 5 to resume.

## 1. Verify the GPU(s)

In [ ]:
!nvidia-smi

## 2. Clone the repo
Onto `/kaggle/working` (persists across restarts within a session).

In [ ]:
import os

REPO_URL = "https://github.com/chinesegorilla99/probe-capacity-invariance.git"
REPO_DIR = "/kaggle/working/probe-capacity-invariance"

if not os.path.exists(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}
else:
    !cd {REPO_DIR} && git pull

%cd {REPO_DIR}

## 3. Install dependencies
Without disturbing Kaggle's preinstalled, CUDA-matched `torch`/`torchvision`.

In [ ]:
!pip install -q -e . --no-deps
!pip install -q h5py

In [ ]:
import torch
print("torch", torch.__version__, "| CUDA:", torch.cuda.is_available(),
      "| device count:", torch.cuda.device_count())
for i in range(torch.cuda.device_count()):
    print(f"  cuda:{i} = {torch.cuda.get_device_name(i)}")

## 4. Download Shapes3D + build the image cache
color_strong is a Shapes3D cell. `--build-cache` decompresses once into an
uncompressed memmap the trainers mmap. Idempotent.

In [ ]:
!cd /kaggle/working/probe-capacity-invariance && python -m src.data.shapes3d --download --build-cache

## 5. Train — color_strong seeds 10, 11 (from scratch)
Launches one `train_simclr` per GPU, each pinned via `CUDA_VISIBLE_DEVICES`,
training from random init to 50 epochs. Per-seed logs in `results/sweep_logs/`.
Idempotent + resumable — **re-run this cell to continue** after a session ends;
done seeds skip, partial seeds resume from their own `last_ckpt.pt`.

In [ ]:
import os, sys, subprocess, time
from pathlib import Path

REPO = Path("/kaggle/working/probe-capacity-invariance")
ENC = REPO / "results" / "encoders"
LOGS = REPO / "results" / "sweep_logs"; LOGS.mkdir(parents=True, exist_ok=True)

CONFIG = "configs/run/color_strong.yaml"
SEEDS = [10, 11]                   # fresh — no checkpoints required
N_GPUS = max(1, torch.cuda.device_count())

def run_id(seed): return f"color_strong_seed{seed}"
def is_done(seed): return (ENC / run_id(seed) / "backbone.pt").exists()

def _launch(seed, gpu):
    rid = run_id(seed)
    env = dict(os.environ,
               CUDA_VISIBLE_DEVICES=str(gpu),
               PYTORCH_CUDA_ALLOC_CONF="expandable_segments:True")
    log = open(LOGS / f"{rid}.log", "a")
    cmd = [sys.executable, "-m", "src.encoders.train_simclr", "--config", CONFIG,
           "--set", f"run.seed={seed}", "run.num_workers=2", "run.device=cuda"]
    p = subprocess.Popen(cmd, cwd=str(REPO), env=env, stdout=log, stderr=subprocess.STDOUT)
    return {"proc": p, "log": log, "rid": rid}

def run(n_gpus=N_GPUS, poll=15):
    subprocess.run(["pkill", "-9", "-f", "src.encoders.train_simclr"])  # reap orphans
    time.sleep(2)
    queue = [s for s in SEEDS if not is_done(s)]
    print(f"{len(SEEDS)} seeds | {len(SEEDS)-len(queue)} done | {len(queue)} to run on {n_gpus} GPU(s)")
    slots = [None] * n_gpus
    try:
        while queue or any(slots):
            for g in range(n_gpus):
                if slots[g] is None and queue:
                    seed = queue.pop(0)
                    if is_done(seed):
                        continue
                    slots[g] = _launch(seed, g)
                    print(f"[GPU{g}] start  {slots[g]['rid']}", flush=True)
            time.sleep(poll)
            for g in range(n_gpus):
                s = slots[g]
                if s and s["proc"].poll() is not None:
                    s["log"].close()
                    seed = int(s["rid"].split("seed")[1])
                    ok = is_done(seed)
                    print(f"[GPU{g}] {'DONE ' if ok else 'EXIT(rc=%s) ' % s['proc'].returncode}{s['rid']}", flush=True)
                    slots[g] = None
    except KeyboardInterrupt:
        for s in slots:
            if s: s["proc"].terminate()
        print("interrupted — checkpoints are saved; re-run this cell to resume")
        return
    print("done — both seeds have a backbone.pt")

run()

## 6. Monitor
Re-run any time while section 5 is training.

In [ ]:
!nvidia-smi --query-gpu=index,name,utilization.gpu,memory.used,memory.total,temperature.gpu --format=csv

## 7. Progress + final diagnostics
Per-seed state and, once done, the collapse diagnostics for each encoder.

In [ ]:
import json as _json

for seed in [10, 11]:
    rid = f"color_strong_seed{seed}"; d = ENC / rid
    if (d / "backbone.pt").exists():
        m = _json.loads((d / "metrics.json").read_text()); diag = m.get("diagnostics", {})
        print(f"{rid}: DONE loss={m['final_loss']:.4f} nan={m['nan_aborted']} epochs={m['epochs_run']} | "
              f"feat_std={diag.get('feat_std'):.4f} eff_rank={diag.get('eff_rank'):.1f} "
              f"align={diag.get('alignment'):.3f} unif={diag.get('uniformity'):.3f}")
    elif (d / "last_ckpt.pt").exists():
        last = None
        for line in (d / "train_log.jsonl").read_text().splitlines():
            try: last = _json.loads(line).get("epoch", last)
            except Exception: pass
        print(f"{rid}: partial @ep{last}")
    else:
        print(f"{rid}: todo")

## 8. Save the finished encoders
**Save Version (Commit)** to persist `/kaggle/working`. The two `backbone.pt`
under `results/encoders/color_strong_seed{10,11}/` are the deliverable — with
these, all 12 color_strong seeds exist and the arm is ready for the probing step.